# META-CXR — Paper Table 6 (Report-Generation BERTScore across 5 common abnormalities)

Reproduces **Table 6** of the IEEE Access paper: mean F1 over the 5 common
abnormalities (Atelectasis, Cardiomegaly, Consolidation, Edema, Pleural Effusion)
for each vision-encoder configuration on the **MIMIC-CXR test set**.

| Paper label | Encoder in this repo |
|-------------|----------------------|
| RN50 | BioViL-T (CNN backbone) |
| ViT  | PubMedCLIP |
| Swin | Swin Transformer |

**Method.** A single model is trained with all three encoders (`07_all_three`).
At inference we toggle encoder streams on/off inside MHCAC (passing `None` for
disabled streams), so every row uses the *same* weights and expert tokens — only
the active encoder set changes. This matches the paper's encoder-ablation setup.

**Prerequisites**
- Accelerator: **GPU** (T4 or P100). Internet **ON** (frozen backbones download from HF).
- Kaggle Secret **`GCS_SERVICE_ACCOUNT`** — service-account JSON (or base64) with read
  access to `gs://meta-cxr-checkpoint` (holds `07_all_three/checkpoint_best.pth`).
- Three datasets attached under `/kaggle/input/datasets/phuong20052/`:
  `mimic-cxr-jpg-lite`, `mimic-cxr-reported`, `mimic-cxr-p10-processed`.

Run cells top to bottom.

## Cell 0 — Load Kaggle Secret (`GCS_SERVICE_ACCOUNT`)

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["GCS_SERVICE_ACCOUNT"] = user_secrets.get_secret("GCS_SERVICE_ACCOUNT")
print("Kaggle secret loaded: GCS_SERVICE_ACCOUNT")

## Cell 1 — Install Dependencies

Same dependency set as the training notebook so the LAVIS / BioViL-T / Swin model
stack builds identically.

In [ ]:
import subprocess, sys

packages = [
    "omegaconf==2.3.0",
    "pycocoevalcap",
    "scikit-image",
    "torchinfo",
    "loralib==0.1.1",
    "iterative-stratification",
    "iopath",
    "hi-ml-multimodal",       # provides health_multimodal used by biovil_t
    "timm>=0.9.0",            # required by the Swin encoder
    "spacy",
    "nltk>=3.9",
    "google-cloud-storage",
    "bert-score",        # Table 6 BERTScore metric
    "transformers==4.44.2",   # pin for Qformer.py compatibility
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)

# peft at the exact commit used by the project
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "git+https://github.com/huggingface/peft.git@e536616888d51b453ed354a6f1e243fecb02ea08"],
    check=True,
)

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)

import torch
print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

## Cell 2 — Clone Repository

In [ ]:
import os

REPO_DIR = "/kaggle/working/META-CXR"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/minhphuong150505/Meta-CXR-Kaggle.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## Cell 3 — Verify Kaggle Datasets & Stage Reports CSV

Checks the three mounted datasets and copies `mimic_cxr_cleaned.csv` to
`/kaggle/working/` (the path the dataset loader expects).

In [ ]:
import os
import shutil
import pandas as pd

KAGGLE_INPUT   = "/kaggle/input/datasets/phuong20052/mimic-cxr-jpg-lite"
IMAGE_ROOT     = KAGGLE_INPUT
REPORTS_ROOT   = "/kaggle/input/datasets/phuong20052/mimic-cxr-reported"
PROCESSED_ROOT = "/kaggle/input/datasets/phuong20052/mimic-cxr-p10-processed"

REQUIRED_CSVS = [
    "mimic-cxr-2.0.0-split.csv",
    "mimic-cxr-2.0.0-chexpert.csv",
    "mimic-cxr-2.0.0-metadata.csv",
]
PROCESSED_REQUIRED = ["train.csv", "val.csv", "test.csv"]
CLEANED_CSV   = "mimic_cxr_cleaned.csv"
REPORTS_LOCAL = "/kaggle/working/mimic_cxr_cleaned.csv"

for name, path in {"KAGGLE_INPUT": KAGGLE_INPUT, "REPORTS_ROOT": REPORTS_ROOT,
                   "PROCESSED_ROOT": PROCESSED_ROOT}.items():
    if not os.path.isdir(path):
        raise FileNotFoundError(f"{name} not found: {path}")

for fname in REQUIRED_CSVS:
    if not os.path.exists(os.path.join(KAGGLE_INPUT, fname)):
        raise FileNotFoundError(f"Missing metadata CSV: {fname}")
for fname in PROCESSED_REQUIRED:
    if not os.path.exists(os.path.join(PROCESSED_ROOT, fname)):
        raise FileNotFoundError(f"Missing preprocessed CSV: {fname}")

csv_in_dataset = os.path.join(REPORTS_ROOT, CLEANED_CSV)
if not os.path.exists(csv_in_dataset):
    raise FileNotFoundError(f"{CLEANED_CSV} not found at {csv_in_dataset}")
os.makedirs(os.path.dirname(REPORTS_LOCAL), exist_ok=True)
if not os.path.exists(REPORTS_LOCAL) or os.path.getsize(REPORTS_LOCAL) != os.path.getsize(csv_in_dataset):
    shutil.copy2(csv_in_dataset, REPORTS_LOCAL)

for k, v in {
    "KAGGLE_INPUT": KAGGLE_INPUT, "IMAGE_ROOT": IMAGE_ROOT,
    "REPORTS_ROOT": REPORTS_ROOT, "PROCESSED_ROOT": PROCESSED_ROOT,
    "REPORTS_CSV": REPORTS_LOCAL,
}.items():
    os.environ[k] = v

print("All datasets present.")
print(f"Reports CSV: {REPORTS_LOCAL} ({len(pd.read_csv(REPORTS_LOCAL))} rows)")

## Cell 4 — Write `configs/env_config.yaml`

In [ ]:
import os
import subprocess

result = subprocess.run("readlink -f $(which java) | sed 's|/bin/java||'",
                        shell=True, capture_output=True, text=True)
java_home = result.stdout.strip() or "/usr/lib/jvm/java-8-openjdk-amd64/jre"
java_path = java_home + "/bin:"

KAGGLE_INPUT   = os.environ["KAGGLE_INPUT"]
IMAGE_ROOT     = os.environ["IMAGE_ROOT"]
REPORTS_CSV    = os.environ["REPORTS_CSV"]
PROCESSED_ROOT = os.environ["PROCESSED_ROOT"]

env_config_content = f"""paths:
  data_root: \"{KAGGLE_INPUT}\"
  mimic_cxr_jpg_root: \"{IMAGE_ROOT}\"
  split_csv: \"{KAGGLE_INPUT}/mimic-cxr-2.0.0-split.csv\"
  reports_csv: \"{REPORTS_CSV}\"
  chexpert_csv: \"{KAGGLE_INPUT}/mimic-cxr-2.0.0-chexpert.csv\"
  metadata_csv: \"{KAGGLE_INPUT}/mimic-cxr-2.0.0-metadata.csv\"
  processed_dir: \"{PROCESSED_ROOT}\"
  processed_train_csv: \"{PROCESSED_ROOT}/train.csv\"
  processed_val_csv: \"{PROCESSED_ROOT}/val.csv\"
  processed_test_csv: \"{PROCESSED_ROOT}/test.csv\"
  output_dir: \"/kaggle/temp/output\"
  checkpoint_dir: \"/kaggle/temp/checkpoints\"
  gcs_bucket: \"gs://meta-cxr-checkpoint\"
  gcs_project: \"mimic-cxr-jpg-491409\"

wandb:
  entity: \"phuongnm150505-uit\"
  project: \"meta-cxr-encoder-comparison\"

java:
  home: \"{java_home}\"
  path: \"{java_path}\"
"""

os.makedirs("configs", exist_ok=True)
with open("configs/env_config.yaml", "w") as f:
    f.write(env_config_content)
print(env_config_content)

## Cell 5 — Download `07_all_three/checkpoint_best.pth` from GCS

Pulls the all-encoders checkpoint from `gs://meta-cxr-checkpoint/07_all_three/`
using the `GCS_SERVICE_ACCOUNT` secret. Hard-fails with a clear message if the
checkpoint is not present in the bucket.

In [ ]:
import base64
import json
import os
from pathlib import Path

GCS_PROJECT = "mimic-cxr-jpg-491409"
GCS_BUCKET  = "meta-cxr-checkpoint"
GCS_PREFIX  = "07_all_three"
CKPT_NAME   = "checkpoint_best.pth"
LOCAL_CKPT  = Path("/kaggle/temp/checkpoints/07_all_three/checkpoint_best.pth")


def _load_service_account_info():
    raw = os.environ.get("GCS_SERVICE_ACCOUNT")
    if not raw:
        raise RuntimeError("GCS_SERVICE_ACCOUNT secret not set (run Cell 0).")
    raw = raw.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return json.loads(base64.b64decode(raw).decode("utf-8"))


from google.cloud import storage
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_info(_load_service_account_info())
client = storage.Client(project=GCS_PROJECT, credentials=credentials)

blob = client.bucket(GCS_BUCKET).blob(f"{GCS_PREFIX}/{CKPT_NAME}")
if not blob.exists():
    raise FileNotFoundError(
        f"gs://{GCS_BUCKET}/{GCS_PREFIX}/{CKPT_NAME} not found. "
        "Train/upload 07_all_three first, or check the service-account bucket access."
    )

LOCAL_CKPT.parent.mkdir(parents=True, exist_ok=True)
blob.download_to_filename(str(LOCAL_CKPT))
size_mb = LOCAL_CKPT.stat().st_size / (1024 ** 2)
print(f"Downloaded gs://{GCS_BUCKET}/{GCS_PREFIX}/{CKPT_NAME} -> {LOCAL_CKPT} ({size_mb:.1f} MB)")

## Cell 6 — Write the Table 6 Evaluator

Writes `eval_encoder_toggle_table6.py` to the repo. It keeps the trained `07_all_three`
weights, then toggles encoder streams feeding **both** the MHCAC findings and the
Q-Former generation tokens, generates the Findings section with Vicuna-7B + LoRA, and
computes mean **BERTScore** (deberta-xlarge-mnli) over a 300-sample test subset for each
encoder configuration.


In [ ]:
%%writefile eval_encoder_toggle_table6.py
#!/usr/bin/env python3
"""Table 6: report-generation BERTScore under encoder toggling from 07_all_three.

Loads the single ``07_all_three`` checkpoint (same one Table 5 uses), then for
each encoder configuration masks the image streams that feed BOTH the MHCAC
findings and the Q-Former generation tokens -- only the selected encoder's
features pass through to the generation head, while MHCAC / soft-prompt tokens /
LLM weights stay unchanged (paper's stated Table 6 methodology). It generates the
Findings section with Vicuna-7B + LoRA and scores against references with
BERTScore. The test set is subsampled (default 300) to fit one Kaggle session.

This mirrors evaluation/eval_encoder_toggle_07.py (Table 5, classification) for
the toggle/checkpoint logic, and evaluation/eval_bertscore_vicuna.py for the
Vicuna generation + BERTScore logic. No model code is modified: the per-stream
Q-Former projection of _encode_image_streams is replicated here with masking.
"""

from __future__ import annotations

import argparse
import gc
import json
import random
import sys
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm
from transformers import LlamaTokenizer
from peft import PeftModelForCausalLM

import model.lavis.tasks as tasks
from model.lavis.common.config import Config
from model.lavis.common.registry import registry

# Registration imports required by the LAVIS registry.
from model.lavis.common.optims import LinearWarmupCosineLRScheduler, LinearWarmupStepLRScheduler  # noqa: F401
from model.lavis.datasets.builders import *  # noqa: F401,F403
from model.lavis.models import *  # noqa: F401,F403
from model.lavis.processors import *  # noqa: F401,F403
from model.lavis.tasks import *  # noqa: F401,F403
from model.lavis.data.ReportDataset import MIMIC_CXR_Dataset
from model.lavis.models.blip2_models.modeling_llama_imgemb import LlamaForCausalLM  # noqa: F401
from local_config import VIS_ROOT

registry.mapping["paths"]["cache_root"] = "."

# --- Vicuna / generation constants (from eval_bertscore_vicuna.py) -----------
VICUNA_MODEL_ID = "lmsys/vicuna-7b-v1.3"
NUM_IMG_TOKENS = 32
IMG_TOKEN = "<IMG>"
IMG_TOKEN_BLOCK = IMG_TOKEN * NUM_IMG_TOKENS
MAX_NEW_TOKENS = 300
NUM_BEAMS = 1
SEED = 16

ABNORMALITIES_14 = [
    "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia", "Atelectasis",
    "Pneumothorax", "Pleural Effusion", "Pleural Other", "Fracture", "Support Devices",
]
CLASS_MAP = {"negative": 0, "positive": 1, "uncertain": 2}

# Encoder configurations (paper Table 6). pubmedclip_swin has no paper value,
# matching Table 5 (eval_encoder_toggle_07.py).
TOGGLE_RUNS = [
    {"run": "01_biovil_only",       "RN50": True,  "ViT": False, "Swin": False},
    {"run": "02_pubmedclip_only",   "RN50": False, "ViT": True,  "Swin": False},
    {"run": "03_swin_only",         "RN50": False, "ViT": False, "Swin": True},
    {"run": "04_biovil_pubmedclip", "RN50": True,  "ViT": True,  "Swin": False},
    {"run": "05_biovil_swin",       "RN50": True,  "ViT": False, "Swin": True},
    {"run": "06_pubmedclip_swin",   "RN50": False, "ViT": True,  "Swin": True},
    {"run": "07_all_three",         "RN50": True,  "ViT": True,  "Swin": True},
]

PAPER_BERTSCORE = {
    "01_biovil_only": 0.312,
    "02_pubmedclip_only": 0.289,
    "03_swin_only": 0.267,
    "04_biovil_pubmedclip": 0.401,
    "05_biovil_swin": 0.394,
    "06_pubmedclip_swin": None,
    "07_all_three": 0.426,
}

# raddino is disabled for 07_all_three, so its newly-initialized alignment head
# is never used at inference and is legitimately absent from the checkpoint.
ALLOWED_MISSING_PREFIXES = (
    "visual_encoder.",
    "pubmedclip.model.",
    "swin.model.",
    "mhcac.embedding_alignment.raddino_",
)

PROMPT_TEMPLATE = (
    "A chat between a curious user and an artificial intelligence assistant."
    "The assistant gives professional, detailed, and polite answers to the user's questions. "
    "USER: Image information: {img_block}.\n\n"
    "Abnormality information: {findings}\n\n"
    "Act as an expert radiologist. Using only the structured abnormality information and the image-derived features above, "
    "write the *Findings* section of a chest X-ray report.\n\n"
    "- Do not invent findings. Only describe abnormalities explicitly provided in the 'Abnormality information'.\n"
    "- Do not repeat the same information using different wording.\n"
    "- Use a single, fluent paragraph in formal radiological style.\n"
    "- Use cautious and precise language if uncertain abnormalities are present.\n"
    "- Avoid enumeration, bullet points, and speculative phrases.\n"
    "- The report should reflect the clinical tone and structure of professionally written reports.\n\n"
    "Return only the generated findings text. ASSISTANT:"
)

BERTSCORE_MODEL = "microsoft/deberta-xlarge-mnli"

_LLM_CACHE: dict = {}


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--project-dir", type=Path, default=Path(__file__).resolve().parent.parent)
    parser.add_argument(
        "--cfg",
        type=Path,
        default=Path("pretraining/configs/encoder_comparison/07_all_three.yaml"),
    )
    parser.add_argument(
        "--checkpoint",
        type=Path,
        default=Path("/kaggle/temp/checkpoints/07_all_three/checkpoint_best.pth"),
    )
    parser.add_argument("--limit", type=int, default=300, help="Subsample N test samples.")
    parser.add_argument("--batch-size", type=int, default=1)
    parser.add_argument("--num-workers", type=int, default=2)
    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument(
        "--bertscore-device",
        default="cpu",
        help="Device for BERTScore (cpu keeps GPU memory for Vicuna).",
    )
    parser.add_argument("--out-dir", type=Path, default=Path("output/encoder_toggle_table6"))
    parser.add_argument("--allow-frozen-missing", action=argparse.BooleanOptionalAction, default=True)
    return parser.parse_args()


def seed_everything(device: str) -> None:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if device == "cuda":
        torch.cuda.manual_seed_all(SEED)


def load_torch_checkpoint(path: Path):
    try:
        return torch.load(str(path), map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(str(path), map_location="cpu")


def build_cfg(cfg_path: Path) -> Config:
    args = SimpleNamespace(cfg_path=str(cfg_path), options=None)
    return Config(args)


def validate_load_result(missing, unexpected, allow_frozen_missing) -> None:
    if unexpected:
        raise RuntimeError(f"Unexpected checkpoint keys: {unexpected[:20]}")
    if allow_frozen_missing:
        invalid = [k for k in missing if not k.startswith(ALLOWED_MISSING_PREFIXES)]
        if invalid:
            raise RuntimeError(f"Checkpoint missing non-frozen/non-backbone keys: {invalid[:20]}")
    elif missing:
        raise RuntimeError(f"Checkpoint is missing keys: {missing[:20]}")


def build_model(cfg: Config, checkpoint_path: Path, device: str, allow_frozen_missing: bool):
    task = tasks.setup_task(cfg)
    model = task.build_model(cfg)
    ckpt = load_torch_checkpoint(checkpoint_path)
    state_dict = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f"Loaded {checkpoint_path}; missing={len(missing)}, unexpected={len(unexpected)}")
    validate_load_result(list(missing), list(unexpected), allow_frozen_missing)
    model.to(device)
    # Pubmedclip hardcodes self.device='cuda' (=cuda:0) at init and uses it to
    # place its inputs in forward; realign with the actual device so it works
    # when the model is placed on cuda:1.
    if getattr(model, "pubmedclip", None) is not None:
        model.pubmedclip.device = device
    model.eval()
    return model


def make_test_loader(cfg: Config, batch_size: int, num_workers: int, limit: int | None, device: str):
    dataset = MIMIC_CXR_Dataset(
        vis_processor=None,
        text_processor=None,
        vis_root=VIS_ROOT,
        split="test",
        cfg=cfg,
        truncate=None,
    )
    if limit is not None:
        dataset = Subset(dataset, list(range(min(limit, len(dataset)))))
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(device == "cuda"),
    )


@torch.no_grad()
def encode_toggled(model, image, use_rn50: bool, use_vit: bool, use_swin: bool):
    """Replicate forward_image with per-encoder masking.

    Only the selected encoders' Q-Former projections are concatenated and fed to
    the Q-Former; the same selection masks the MHCAC streams. Returns toggled
    (classification_logits, qformer_embeds).
    """
    streams = []
    cnn_patches = vit_patches = swin_patches = None

    if model.use_biovil:
        cnn_raw = model.visual_encoder(image).projected_patch_embeddings.reshape(
            image.shape[0], -1, 1408
        )
        cnn_patches = model.ln_vision(cnn_raw)
        if use_rn50:
            streams.append(cnn_patches)

    if model.use_pubmedclip:
        vit_patches, pubmed_projection = model.pubmedclip(image, apply_aug=False)
        if use_vit:
            streams.append(pubmed_projection)

    if model.use_swin:
        swin_patches = model.swin(image)
        if use_swin:
            streams.append(model.swin_qformer_proj(swin_patches))

    if not streams:
        raise ValueError("No encoder stream selected for this configuration.")

    concat_image_embeds = torch.cat(streams, dim=1)

    cls_logits, _, _, _, _ = model.mhcac(
        cnn_patches=cnn_patches if use_rn50 else None,
        vit_patches=vit_patches if use_vit else None,
        swin_patches=swin_patches if use_swin else None,
        raddino_patches=None,
        text_embeddings=None,
        labels=None,
    )

    image_atts = torch.ones(concat_image_embeds.size()[:-1], dtype=torch.long, device=image.device)
    query_tokens = model.query_tokens.expand(concat_image_embeds.shape[0], -1, -1)
    query_output = model.Qformer.bert(
        query_embeds=query_tokens,
        encoder_hidden_states=concat_image_embeds,
        encoder_attention_mask=image_atts,
        return_dict=True,
    )
    return cls_logits.float().cpu(), query_output.last_hidden_state.float().cpu()


def classify_with_thresholds(logits, thresholds):
    assert logits.shape == (14, 3), f"expected (14, 3), got {tuple(logits.shape)}"
    probs = torch.softmax(logits, dim=-1).tolist()
    out = {"positive": [], "negative": [], "uncertain": []}
    for abn, p in zip(ABNORMALITIES_14, probs):
        if abn == "No Finding":
            continue
        thresholds_abn = thresholds.get(abn, {})
        best_cls, best_score = None, 0.0
        for cls_name, cls_idx in CLASS_MAP.items():
            threshold = thresholds_abn.get(cls_name, 0.5)
            prob = p[cls_idx]
            if prob >= threshold and prob > best_score:
                best_cls = cls_name
                best_score = prob
        if best_cls is not None:
            out[best_cls].append(abn)
    return out


def format_findings_dict(classifications):
    parts = []
    for label in ["positive", "negative", "uncertain"]:
        items = classifications[label]
        if items:
            parts.append(f"{label.capitalize()} findings: {', '.join(items)}")
    return ". ".join(parts) if parts else "no common findings"


def build_prompt(classifications):
    findings = format_findings_dict(classifications)
    return PROMPT_TEMPLATE.format(img_block=IMG_TOKEN_BLOCK, findings=findings)


def get_vicuna(project_dir: Path):
    if "model" in _LLM_CACHE:
        return _LLM_CACHE["model"], _LLM_CACHE["tokenizer"]

    lora_path = project_dir / "checkpoints" / "lora-vicuna-7b-report-20250621"
    print(f">>> Loading {VICUNA_MODEL_ID} ...")
    tokenizer = LlamaTokenizer.from_pretrained(
        VICUNA_MODEL_ID, use_fast=False, truncation_side="left", padding_side="left"
    )
    # Load the whole LLM on a single GPU (cuda:0). Sharding across 2 GPUs with
    # device_map="auto" puts the image-embedding injection cat() on mixed
    # devices (cuda:0 vs cuda:1) and crashes; single-device matches the original
    # single-GPU (L4) run.
    base = LlamaForCausalLM.from_pretrained(
        VICUNA_MODEL_ID, torch_dtype=torch.float16, device_map={"": 0}
    )
    tokenizer.pad_token = tokenizer.unk_token
    base.base_model.img_proj_layer = nn.Linear(
        768, base.base_model.config.hidden_size
    ).to(base.base_model.device)
    tokenizer.add_special_tokens({"additional_special_tokens": [IMG_TOKEN]})

    print(f">>> Attaching LoRA from {lora_path} ...")
    llm = PeftModelForCausalLM.from_pretrained(
        base, str(lora_path), torch_dtype=torch.float16, use_ram_optimized_load=False
    ).half()
    llm.eval()

    _LLM_CACHE["model"] = llm
    _LLM_CACHE["tokenizer"] = tokenizer
    return llm, tokenizer


@torch.no_grad()
def generate_report(prompt, qformer_embs, llm, tokenizer):
    assert qformer_embs.dim() == 3 and qformer_embs.shape[1] == NUM_IMG_TOKENS, (
        f"qformer_embs must be (B, {NUM_IMG_TOKENS}, 768), got {tuple(qformer_embs.shape)}"
    )
    # modeling_llama_imgemb reads the image embeddings from this side-channel file.
    torch.save(qformer_embs, "current_chat_img.pt")
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs["input_ids"].to(llm.device)
    out = llm.generate(
        input_ids=input_ids,
        dicom=None,
        use_img=True,
        return_dict_in_generate=True,
        output_scores=False,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS,
        do_sample=False,
    )
    preds = tokenizer.batch_decode(out.sequences, skip_special_tokens=True)
    return preds[0].split("ASSISTANT:")[-1].strip()


def compute_bertscore(predictions, references, device):
    from bert_score import score as bert_score_fn

    if not predictions:
        return float("nan")
    _P, _R, F1 = bert_score_fn(
        predictions,
        references,
        lang="en",
        model_type=BERTSCORE_MODEL,
        rescale_with_baseline=False,
        verbose=False,
        device=device,
    )
    return float(F1.mean().item())


def main() -> None:
    args = parse_args()
    project_dir = args.project_dir.resolve()
    cfg_path = args.cfg if args.cfg.is_absolute() else project_dir / args.cfg
    checkpoint_path = args.checkpoint if args.checkpoint.is_absolute() else project_dir / args.checkpoint
    out_dir = args.out_dir if args.out_dir.is_absolute() else project_dir / args.out_dir
    out_dir.mkdir(parents=True, exist_ok=True)

    seed_everything(args.device)
    with open(project_dir / "threshold.json") as f:
        thresholds = json.load(f)

    print("project_dir =", project_dir)
    print("cfg_path    =", cfg_path)
    print("checkpoint  =", checkpoint_path)
    print("device      =", args.device, "| bertscore_device =", args.bertscore_device)
    print("limit       =", args.limit)

    # Vicuna is loaded whole on cuda:0; on 2+ GPUs put the meta-cxr encoder/Q-Former
    # model on cuda:1 to avoid co-location OOM. Its outputs are moved to CPU in
    # encode_toggled, so the LLM device is independent.
    meta_device = args.device
    if args.device == "cuda" and torch.cuda.device_count() >= 2:
        meta_device = "cuda:1"
    print("meta_device =", meta_device, "| vicuna on cuda:0")

    cfg = build_cfg(cfg_path)
    model = build_model(cfg, checkpoint_path, meta_device, args.allow_frozen_missing)
    loader = make_test_loader(cfg, args.batch_size, args.num_workers, args.limit, meta_device)
    n_samples = len(loader.dataset)
    print("test samples =", n_samples)

    llm, tokenizer = get_vicuna(project_dir)

    rows = []
    details = {}
    for item in TOGGLE_RUNS:
        run_name = item["run"]
        print(f"\n{'='*60}\n{run_name}\n{'='*60}")
        predictions, references = [], []
        for batch in tqdm(loader, desc=f"{run_name} gen"):
            image = batch["image"].to(meta_device, non_blocking=True)
            cls_logits, qformer_embs = encode_toggled(
                model, image, item["RN50"], item["ViT"], item["Swin"]
            )
            for i in range(cls_logits.shape[0]):
                classifications = classify_with_thresholds(cls_logits[i], thresholds)
                prompt = build_prompt(classifications)
                try:
                    pred = generate_report(prompt, qformer_embs[i : i + 1], llm, tokenizer)
                except Exception as exc:  # noqa: BLE001 - keep going, record empty pred
                    print(f"  generate failed for sample {len(predictions)}: {exc}")
                    pred = ""
                ref_field = batch["text_output"]
                ref = ref_field[i] if isinstance(ref_field, (list, tuple)) else str(ref_field[i])
                predictions.append(pred)
                references.append(ref)

        mean_f1 = compute_bertscore(predictions, references, args.bertscore_device)
        paper = PAPER_BERTSCORE[run_name]
        rows.append(
            {
                "Run": run_name,
                "RN50": "yes" if item["RN50"] else "no",
                "ViT": "yes" if item["ViT"] else "no",
                "Swin": "yes" if item["Swin"] else "no",
                "BERTScore": round(mean_f1, 4),
                "N_samples": len(predictions),
                "Paper BERTScore": paper if paper is not None else None,
                "Delta vs Paper": round(mean_f1 - paper, 4) if paper is not None else None,
            }
        )
        details[run_name] = {"bertscore_f1": mean_f1, "n_samples": len(predictions)}

        out_jsonl = out_dir / f"reports_{run_name}.jsonl"
        with out_jsonl.open("w", encoding="utf-8") as f:
            for p, r in zip(predictions, references):
                f.write(json.dumps({"pred": p, "ref": r}) + "\n")
        print(f">>> {run_name}: BERTScore F1={mean_f1:.4f} ({len(predictions)} samples) -> {out_jsonl}")

    table = pd.DataFrame(rows)
    table_path = out_dir / "table6_bertscore_table.csv"
    json_path = out_dir / "table6_bertscore_details.json"
    table.to_csv(table_path, index=False)
    with json_path.open("w", encoding="utf-8") as f:
        json.dump(
            {
                "checkpoint": str(checkpoint_path),
                "cfg": str(cfg_path),
                "limit": args.limit,
                "num_samples": n_samples,
                "llm": VICUNA_MODEL_ID,
                "bertscore_model": BERTSCORE_MODEL,
                "max_new_tokens": MAX_NEW_TOKENS,
                "num_beams": NUM_BEAMS,
                "details": details,
            },
            f,
            indent=2,
        )

    print("\nTable 6:")
    print(table.to_string(index=False))
    print("\nWrote:", table_path)
    print("Wrote:", json_path)

    del model, loader
    gc.collect()
    if args.device == "cuda":
        torch.cuda.empty_cache()


if __name__ == "__main__":
    main()


## Cell 7 — Run the Evaluation

Single pass over the MIMIC-CXR test split; encoder streams are cached once and
reused across all configurations. Writes `output/encoder_toggle_07/*`.

In [ ]:
# Stream output directly so the real traceback is visible (subprocess+check hides it).
!cd /kaggle/working/META-CXR && python eval_encoder_toggle_table6.py \
    --project-dir /kaggle/working/META-CXR \
    --checkpoint /kaggle/temp/checkpoints/07_all_three/checkpoint_best.pth \
    --limit 300


## Cell 8 — Table 5

Loads the evaluator output and renders the table. `Mean F1 Score` is this run's
weighted-F1; `Paper F1` is the published value; `Delta vs Paper` is the difference.
(The ViT+Swin row has no paper value — it is not reported in the paper's Table 5.)

In [ ]:
import pandas as pd
df = pd.read_csv(
    "/kaggle/working/META-CXR/output/encoder_toggle_table6/table6_bertscore_table.csv"
)
df
